# Popularityと共通評価基盤の動作確認

架空の6曲・5要求を使い、PythonのRunnerで推薦と評価を実行します。
この結果は基盤の動作確認用で、音楽推薦の研究上の性能を示すものではありません。

リポジトリのルートで `uv sync --locked --group notebook`、
`uv run --locked --group notebook jupyter lab` を実行してこのNotebookを開いてください。
カーネルを再起動し、全セルを上から実行します。
推薦・指標・保存処理は `src/music_recsys/` にあり、CLIと共有しています。

In [ ]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

from music_recsys.data import load_dataset
from music_recsys.evaluation.benchmark import (
    format_summary,
    load_config,
    run_experiment,
    save_report,
)

ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "configs/toy.toml").is_file()
)
CONFIG_PATH = ROOT / "configs/toy.toml"
print("Python:", sys.version.split()[0])
print("Config:", CONFIG_PATH)

## 1. 入力と分割を確認する

学習インタラクションと評価正解は、JSON内で別々に定義しています。
履歴は学習データから用意し、そのユーザーの履歴曲を候補から除きます。
正解はEvaluatorが参照し、推薦関数には渡しません。
seedは記録用で、現在の手動分割とPopularityには乱数処理はありません。

In [ ]:
config = load_config(CONFIG_PATH)
dataset = load_dataset(config.dataset_path, top_k=config.top_k)
display(
    {
        "dataset": dataset.dataset_id,
        "catalog_size": len(dataset.catalog_ids),
        "training_events": len(dataset.train_interactions),
        "requests": len(dataset.requests),
        "top_k": config.top_k,
        "seed": config.seed,
        "dataset_sha256": dataset.sha256,
    }
)

## 2. Pythonの共通Runnerを実行する

`run_experiment` が設定を読み、Popularityを学習して共通Runnerを呼びます。
人気度は学習内の楽曲の出現回数で、同点なら楽曲ID順です。
学習時間と要求ごとの推薦時間を別に記録します。

In [ ]:
report = run_experiment(CONFIG_PATH)
display(Markdown(format_summary(report)))

## 3. 要求ごとの推薦と評価を確認する

r1は2曲の正解のうち1曲を推薦するためRecall@2が0.5です。
r3は正解が2位であるためNDCG@2が約0.630930です。
r4は履歴のないユーザー、r5は正解のない要求です。
正解のない要求の指標はnullとして平均から除外し、対象外件数を記録します。
実行に失敗した要求は、正解があれば0として平均に含めます。

In [ ]:
truth_by_id = {truth.request_id: truth for truth in dataset.ground_truth}
rows = [
    "| Request | User | Recommended | Relevant | NDCG | Recall | Status |",
    "| --- | --- | --- | --- | --- | --- | --- |",
]
for run in report.runs:
    ndcg = "対象外" if run.evaluation.ndcg is None else f"{run.evaluation.ndcg:.6f}"
    recall = "対象外" if run.evaluation.recall is None else f"{run.evaluation.recall:.6f}"
    relevant = truth_by_id[run.request.request_id].relevant_item_ids
    rows.append(
        f"| {run.request.request_id} | {run.request.user_id} | "
        f"{', '.join(run.response.item_ids) or 'なし'} | "
        f"{', '.join(relevant) or 'なし'} | {ndcg} | {recall} | {run.response.status} |"
    )
display(Markdown("\n".join(rows)))

## 4. 集計値を可視化する

正解がある4要求のmacro平均です。図中のラベルはフォント互換性のため英語で表示します。
合成データの値をモデル間の優位性として解釈しないでください。

In [ ]:
import matplotlib.pyplot as plt

if report.summary.n_evaluated:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    labels = [f"NDCG@{config.top_k}", f"Recall@{config.top_k}"]
    bars = ax.bar(
        labels, [report.summary.ndcg, report.summary.recall], color=["#256b78", "#c77936"]
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel("Macro average")
    ax.set_title("Popularity on synthetic data")
    ax.bar_label(bars, fmt="%.3f", padding=4)
    fig.tight_layout()
    plt.show()
else:
    print("候補内の正解がないため、集計指標は未定義です。")

## 5. 結果と再現用の記録を保存する

`results.json` と `summary.md` を新しい実行ディレクトリへ保存します。
再実行しても既存結果を上書きしません。
生結果の保存先はGit管理から除外されます。共有する考察・要約は別途管理します。
JSONには設定値、データ・ソース・設定のハッシュ、実行日時、時間、要求ごとの結果が含まれます。

In [ ]:
RUN_DIR = save_report(report, ROOT / "experiments/toy/runs")
print("Saved:", RUN_DIR)
print("Results:", RUN_DIR / "results.json")

## 次の検証

このNotebookはCLIと同じ計算を呼びます。
次は実データと分割条件を選び、同じデータ・入力・候補集合・K・指標で別の推薦手法を比較します。
B/Cの導入時には自然言語入力の比較、実行予算、LLM費用とツール実行履歴を追加します。